# Week 2 — Session 3
## Schema Design, Normalization, and Denormalization

## Lab 1 — Identify Entities and Attributes from a Flat File

The flat CSV mixes information about customers, orders, products, and order items.

### Customer
- customer_id
- customer_name
- customer_email
- shipping_address

### Order
- order_id
- order_date
- customer_id
- shipping_address

### Product
- product_id
- product_name
- price
- product_category

### OrderItem
- order_id
- product_id
- quantity
- unit_price

The reason for separating these entities is that customer, order, and product information can repeat across many rows. Splitting them into separate entities reduces duplication and makes the data model easier to maintain.

## Lab 2 — Classify Relationships and Keys

| Entity | Primary Key | Foreign Keys | Relationship Notes |
|---|---|---|---|
| Customer | customer_id | None | One customer can have many orders |
| Order | order_id | customer_id | One order belongs to one customer |
| Product | product_id | None | One product can appear in many order items |
| OrderItem | order_id + product_id | order_id, product_id | Connects orders and products |

### Relationship Explanation

A customer can have many orders.

An order can contain many products.

A product can appear in many different orders.

Because Orders and Products have a many-to-many relationship, the `OrderItem` table is used between them.

Each `OrderItem` row represents one product inside one order.

## Lab 2 — Classify Relationships and Keys

| Entity | Primary Key | Foreign Keys | Relationship Notes |
|---|---|---|---|
| Customer | customer_id | None | One customer can have many orders |
| Order | order_id | customer_id | One order belongs to one customer |
| Product | product_id | None | One product can appear in many order items |
| OrderItem | order_id + product_id | order_id, product_id | Connects orders and products |

### Relationship Explanation

A customer can have many orders.

An order can contain many products.

A product can appear in many different orders.

Because Orders and Products have a many-to-many relationship, the `OrderItem` table is used between them.

Each `OrderItem` row represents one product inside one order.

## Lab 4 — Convert Repeating Groups into Rows

This exercise converts nested product arrays into separate OrderItem rows.

The nested structure is flattened so that each product inside an order becomes its own row.

In [1]:
orders = [
    {
        "order_id": 1003,
        "customer": "Ana",
        "items": [
            {"product_id": 301, "qty": 2}
        ]
    },
    {
        "order_id": 1004,
        "customer": "Ben",
        "items": [
            {"product_id": 302, "qty": 1},
            {"product_id": 303, "qty": 4}
        ]
    }
]

order_items = []

for order in orders:
    for item in order["items"]:
        order_items.append({
            "order_id": order["order_id"],
            "customer": order["customer"],
            "product_id": item["product_id"],
            "qty": item["qty"]
        })

print(order_items)

[{'order_id': 1003, 'customer': 'Ana', 'product_id': 301, 'qty': 2}, {'order_id': 1004, 'customer': 'Ben', 'product_id': 302, 'qty': 1}, {'order_id': 1004, 'customer': 'Ben', 'product_id': 303, 'qty': 4}]


## Lab 5 — Apply Second Normal Form (2NF)

The original table is:

OrderItem(order_id, product_id, product_name, product_price, qty)

The composite primary key is:

`(order_id, product_id)`

However, `product_name` and `product_price` depend only on `product_id`, not on the complete composite key.

This creates a partial dependency.

### Products

| product_id | product_name | product_price |
|---|---|---:|
| 401 | Gizmo | 9.99 |
| 402 | Doodad | 5.00 |

**Primary Key:** `product_id`

### OrderItems

| order_id | product_id | qty |
|---|---|---:|
| 1005 | 401 | 2 |
| 1005 | 402 | 1 |

**Primary Key:** `(order_id, product_id)`

**Foreign Key:** `product_id` references `Products(product_id)`.

By separating product information from OrderItems, the partial dependency is removed and the design reaches Second Normal Form.

## Lab 6 — Apply Third Normal Form (3NF)

The original structure is:

`Customer(customer_id, name, city, zip, state_name)`

There is a transitive dependency because location information depends on the ZIP code.

For example:

`customer_id → zip → city/state_name`

I would separate the data into two tables.

### Customer

| Column | Role |
|---|---|
| customer_id | Primary Key |
| name | Customer attribute |
| zip | Foreign Key |

### Location

| Column | Role |
|---|---|
| zip | Primary Key |
| city | Location attribute |
| state_name | Location attribute |

The `Customer.zip` field references `Location.zip`.

This removes the transitive dependency because the location details are stored separately instead of being repeated for every customer.

## Lab 7 — Design a Normalized OLTP Schema

The following schema separates customers, orders, products, and order items to reduce duplication and enforce relationships.

```sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    customer_email TEXT UNIQUE,
    shipping_address TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    product_price REAL NOT NULL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date DATE NOT NULL,

    FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,

    FOREIGN KEY (order_id)
        REFERENCES orders(order_id),

    FOREIGN KEY (product_id)
        REFERENCES products(product_id)
);


---

## Lab 8 — Markdown cell

```markdown
## Lab 8 — Create a Denormalized Reporting Table

For reporting, frequently used information can be combined into one wider table.

```sql
CREATE TABLE daily_order_report (
    customer_name TEXT,
    order_id INTEGER,
    order_date DATE,
    total_order_value REAL,
    product_list TEXT,
    shipping_city TEXT
);

INSERT INTO daily_order_report
VALUES (
    'Jane Doe',
    1001,
    '2026-06-01',
    59.97,
    'Widget A, Widget B',
    'Amsterdam'
);

This table is denormalized because information that originally comes from several tables is combined into one reporting table.

The benefit is faster and simpler reporting queries.

The trade-off is duplicated data and additional work to keep the reporting table updated.


---

## Lab 9 — Markdown cell

```markdown
## Lab 9 — Design a Star Schema

I choose the grain:

**One row in FactOrders represents one order item.**

This grain allows detailed analysis by product, customer, and date.

### FactOrders

| Column | Role |
|---|---|
| order_item_key | Primary Key |
| customer_key | Foreign Key |
| product_key | Foreign Key |
| date_key | Foreign Key |
| quantity | Measure |
| sales_amount | Measure |

### CustomerDim

| Column |
|---|
| customer_key — Primary Key |
| customer_id |
| customer_name |
| customer_region |
| customer_email |

### ProductDim

| Column |
|---|
| product_key — Primary Key |
| product_id |
| product_name |
| product_category |
| product_price |

### DateDim

| Column |
|---|
| date_key — Primary Key |
| full_date |
| day |
| month |
| year |

### Explanation

The fact table contains measurable business events such as quantity and sales amount.

The dimension tables contain descriptive information that gives context to those measurements.

The model can answer questions such as:

- sales by product category
- sales by customer region
- daily or monthly sales

## Lab 10 — Map Normalized Data to a Star Schema

The following SQL shows how normalized OLTP data could be transformed into fact-table rows.

The grain is one row per order item.

```sql
INSERT INTO FactOrders (
    customer_key,
    product_key,
    date_key,
    quantity,
    sales_amount
)
SELECT
    c.customer_id,
    p.product_id,
    o.order_date,
    oi.quantity,
    oi.quantity * oi.unit_price AS sales_amount
FROM orders AS o

JOIN customers AS c
    ON o.customer_id = c.customer_id

JOIN order_items AS oi
    ON o.order_id = oi.order_id

JOIN products AS p
    ON oi.product_id = p.product_id;


    The transformation:

Connects orders to customers.
Connects orders to order items.
Connects each order item to its product.
Calculates sales_amount as quantity multiplied by unit price.
Produces one fact row for each order item.


---

## Lab 11 — Markdown cell

```markdown
## Lab 11 — Slowly Changing Dimension Type 2

A Slowly Changing Dimension Type 2 keeps historical versions instead of overwriting old information.

### CustomerDim Structure

| Column | Purpose |
|---|---|
| customer_key | Surrogate Primary Key |
| customer_id | Natural customer identifier |
| customer_name | Customer name |
| city | Customer city |
| effective_from | Date this version became valid |
| effective_to | Date this version stopped being valid |
| current_flag | Shows whether this is the current version |

### Example

Customer 900 moved from Oldtown to Newcity on 2026-05-10.

| customer_key | customer_id | city | effective_from | effective_to | current_flag |
|---:|---:|---|---|---|---:|
| 1 | 900 | Oldtown | 2025-01-01 | 2026-05-09 | 0 |
| 2 | 900 | Newcity | 2026-05-10 | NULL | 1 |

The old row is preserved for historical reporting.

Instead of updating the Oldtown row and losing the history, a new row is inserted for Newcity.

## Lab 12 — Denormalization for Query Performance

The reports currently require many joins:

`orders → customers → addresses → products → categories`

Three possible denormalization strategies are:

### Option 1 — Precomputed Aggregate Tables

Example:

`daily_sales_by_category`

The table could already contain:

- date
- category
- total_quantity
- total_revenue

**Benefit:**  
Dashboard queries become very fast because the calculations have already been performed.

**Trade-off:**  
The aggregate table must be refreshed when new data arrives and can temporarily become stale.

---

### Option 2 — Wide Reporting Table

Create one reporting table containing commonly needed information such as:

- order_id
- order_date
- customer_name
- customer_region
- product_name
- product_category
- quantity
- sales_amount

**Benefit:**  
Reports require fewer joins.

**Trade-off:**  
The same descriptive information is duplicated across many rows, which increases storage and maintenance requirements.

---

### Option 3 — Add Selected Redundant Columns

Frequently used attributes such as `customer_region` or `product_category` can be copied directly into a reporting fact table.

**Benefit:**  
Queries can avoid some joins while keeping the table smaller than a fully denormalized wide table.

**Trade-off:**  
The duplicated columns must be kept synchronized with their source data.

---

### Ranking for Read Performance

1. Precomputed aggregate tables
2. Wide reporting tables
3. Selected redundant columns

The best design depends on the query patterns and business requirements.